## Changing directory

In [11]:
import os
%pwd

'c:\\Users\\Aswin\\MLprojects\\Quality_Of_wine(EndtoEnd ML)\\Quality_Of_Wine-EndtoEnd-\\research'

In [12]:
os.chdir("..")

In [15]:
%pwd

'c:\\Users\\Aswin\\MLprojects\\Quality_Of_wine(EndtoEnd ML)\\Quality_Of_Wine-EndtoEnd-'

## Setting up the Dagshub mlflow_tracking uri,username,password

In [2]:
%pip install dagshub

In [1]:
a=1
b=2
a+b

3

In [1]:
import dagshub
dagshub.init(repo_owner='cecsranjethaswinr23-collab', repo_name='Quality_Of_Wine-EndtoEnd-', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as cecsranjethaswinr23-collab

Initialized MLflow to track repo "cecsranjethaswinr23-collab/Quality_Of_Wine-EndtoEnd-"

Repository cecsranjethaswinr23-collab/Quality_Of_Wine-EndtoEnd- initialized!

c:\Users\Aswin\anaconda3\envs\ml_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run caring-shark-457 at: https://dagshub.com/cecsranjethaswinr23-collab/Quality_Of_Wine-EndtoEnd-.mlflow/#/experiments/0/runs/27fc6cc9697d4a1bb73bf91b70e14d2c
🧪 View experiment at: https://dagshub.com/cecsranjethaswinr23-collab/Quality_Of_Wine-EndtoEnd-.mlflow/#/experiments/0


In [ ]:
import mlflow
import os

# Manually set credentials if dagshub.init isn't used
os.environ['MLFLOW_TRACKING_USERNAME'] = "YOUR_DAGSHUB_USERNAME"
os.environ['MLFLOW_TRACKING_PASSWORD'] = "YOUR_DAGSHUB_TOKEN"

mlflow.set_tracking_uri("https://dagshub.com/YOUR_USERNAME/Quality_Of_Wine-EndtoEnd-.mlflow")
mlflow.set_experiment("Connection_Test")

with mlflow.start_run(run_name="Test_Run"):
    mlflow.log_param("status", "it_works")
    print("Run logged. Check DagsHub now!")

## Config Entity

In [ ]:
# config entity

"""
An Entity is a simple Python class (usually a dataclass) that defines the structure of the data needed for this stage.
In this entity the hyperparams for the model training has to be filled too.
"""

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

## Configuartion manager

In [19]:
from Quality_of_Wine.constants import *
from Quality_of_Wine.utils.common import read_yaml, create_directories, save_json

In [6]:
mlflow.get_tracking_uri()

'https://dagshub.com/cecsranjethaswinr23-collab/Quality_Of_Wine-EndtoEnd-.mlflow'

In [ ]:
# configuration manager

"""
Configuration Manager: This is the "Post Office" of your project. It reads your config.yaml and params.yaml files 
and packages that information into the Config Entity described above.


"""

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    """     
    def get_model_evaluation_config: This is a function inside your Configuration Manager. 
    Its only job is to go into your YAML files, grab the evaluation settings, and return the Config Entity.
    """
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri="https://dagshub.com/cecsranjethaswinr23-collab/Quality_Of_Wine-EndtoEnd-.mlflow",
           
        )

        return model_evaluation_config


## Model Evaluation(component)

In [21]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [22]:
# Model evaluation


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    
    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    


    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # Model registry does not work with file store
            """ 
            1. The Logic Check: if tracking_url_type_store != "file":

            The "File" Case: If you are running MLflow locally on your computer, the URI looks 
            like file:///C:/Users/.... In this mode, MLflow just saves data to a folder on your hard drive.

                The "Remote" Case: If you have set your URI to DagsHub (or any remote server), the URI starts with https://.

                The Goal: This check ensures the code doesn't try to "Register" a model on your local computer, which would 
                          cause an error because the Model Registry requires a central database (like DagsHub).
            
            2. The Remote Path: mlflow.sklearn.log_model(..., registered_model_name="ElasticnetModel")

            If the code detects you are connected to DagsHub, it does two massive things:

                Logging the Model: It uploads your actual model file (the .joblib or .pkl file) to DagsHub. This ensures that if your computer crashes, your trained model is safe in the cloud.

                Model Registration: This is the most important part. By adding registered_model_name, you are telling DagsHub: "This isn't just a test; I want this to be Version 1 of my official 'ElasticnetModel'." * In DagsHub, this model will now appear in the Model Registry tab.
            You can then "Stage" this model (e.g., move it from Staging to Production) within the UI.
            
            3. The Local Path: else: mlflow.sklearn.log_model(model, "model")
            If you are just testing your code locally (without DagsHub), it falls into this else block.

                What it does: it saves the model to your local mlruns folder.

                What it omits: It does not try to name or register it. It just keeps a copy of the model attached to that specific "Run ID" so you can look at it later if needed.

            """


            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

    


## Model evaluation Pipeline

In [23]:
# Pipeline

"""  
Pipeline (Model Evaluation Phase): This is the final script that orchestrates the entire flow. 
It calls the Configuration Manager to get the settings, initializes the Evaluation Class, and triggers 
the logging. It is the "Start" button for this specific phase.
Pipeline starts.

~It asks the Configuration Manager for the Config Entity.
~The Manager runs get_model_evaluation_config to read the YAML files.
~The Model Evaluation Class takes that config and:
~Loads the model.
~Runs eval_metrics to see how smart the model is.
~Runs log_into_mlflow to save those results for you to see on DagsHub.

"""

try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-12 17:34:14,293: INFO: common: yaml file: config\config.yaml loaded successfully]


[2026-03-12 17:34:14,320: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-12 17:34:14,333: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-12 17:34:14,346: INFO: common: created directory at: artifacts]
[2026-03-12 17:34:14,352: INFO: common: created directory at: artifacts/model_evaluation]
[2026-03-12 17:34:16,398: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/03/12 17:34:32 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: ElasticnetModel, version 2
Created version '2' of model 'ElasticnetModel'.


## modular coding for model evaluation in src folder

In [ ]:
"""
~Succcessfully ran the MODEL TRAINER part of the project in reseach folder.

~End To End MLOPS Data Science Project Implementation With Deployment(YT Channel)=https://youtu.be/pxk1Fr33-L4?si=YXioCQhqeFOWQ_az

~Now it has to be covered into modular coding.
   the process and where it has to copied is mentioned below

   *Step 01: entity/config_entity.py
            ~ In the entity/config_entity.py copy and paste the code below the entity of model trainer
              code from config entity markdown above, just the class.
            ~ The crct datatype has to be speecified to the entity(class ModelEvaluationConfig)
   
   *Step 02: src/config/configuration.py
            ~ In the src/config/configuration.py copy and paste the def function of the code from Configuration Manager markdown
              above.
            ~ the other needed package must be imported from src/entity.

   *Step 03: create a new file in src/components/model_evaluation.py(create new file)
            ~ In the src/components/model_evaluation.py copy and paste the code from model trainer markdown above with packages
             and import needed packages(save_json is in utils).
   
   *Step 04: create a new file in src/pipelines/stage_05_model_evaluation.py(create new file)
            ~ Before copying the pipeline code, some code have to coded first that is already in src/pipelines/stage_04_model_trainer.ipynb in model project
              template.
            ~ Check the code to be written before.
   
   *Step 05: To test the pipeline from the above step the execution must be done from main.py
            ~ From the code from the src/pipelines/stage_05_model_evaluation.ipynb copy and paste the code with the packages in the main.py.
            ~ Delete the artifacts folder.
            ~ With different parameters the experiments can be done without deleting the artifacts
            ~ The experiment details gets stored in the mlflow UI.
            ~ And execute it in the terminal -python main.py.

"""